# Cell Tracking Submission
This notebook generates and validates the competition submission file.

Attach the competition test data and the tracking source before running all cells. The final file is written to `/kaggle/working/submission.csv`.

In [ ]:
from pathlib import Path
import csv
import sys
import numpy as np
from scipy.ndimage import center_of_mass

KAGGLE_ROOT = Path('/kaggle/input')
RUNNING_ON_KAGGLE = KAGGLE_ROOT.exists()
OUTPUT_PATH = Path('/kaggle/working/submission.csv') if RUNNING_ON_KAGGLE else Path.cwd() / 'submission.csv'
SEARCH_ROOTS = [KAGGLE_ROOT] if RUNNING_ON_KAGGLE else [Path.cwd()]


def load_labels(path):
    if path.is_dir() and path.name.endswith('.zarr'):
        import zarr
        store = zarr.open(str(path), mode='r')
        if hasattr(store, 'shape'):
            return np.asarray(store)
        for key in ('labels', 'masks'):
            if key in store:
                return np.asarray(store[key])
        raise ValueError(f'No labels array found in {path}')
    loaded = np.load(path)
    if isinstance(loaded, np.lib.npyio.NpzFile):
        key = next(iter(loaded.files))
        return loaded[key]
    return loaded


def extract_nodes(labels):
    if labels.ndim == 3:
        labels = labels[None, ...]
    if labels.ndim != 4:
        raise ValueError(f'Expected 3D or 4D labels, got shape {labels.shape}')
    nodes = []
    for timepoint, frame in enumerate(labels):
        for label in np.unique(frame):
            if label == 0:
                continue
            z, y, x = center_of_mass(frame == label)
            nodes.append({'node_id': -1, 't': timepoint, 'z': round(z), 'y': round(y), 'x': round(x)})
    return nodes


def link_nodes(nodes, max_dist_um=7.0, voxel_size=(1.625, 0.40625, 0.40625)):
    edges = []
    by_time = {}
    for node in nodes:
        by_time.setdefault(node['t'], []).append(node)
    for timepoint, sources in by_time.items():
        targets = by_time.get(timepoint + 1, [])
        candidates = []
        for source in sources:
            for target in targets:
                distance = sum(((source[key] - target[key]) * scale) ** 2 for key, scale in zip(('z', 'y', 'x'), voxel_size)) ** 0.5
                if distance <= max_dist_um:
                    candidates.append((distance, source['node_id'], target['node_id']))
        used_sources = set()
        used_targets = set()
        for _, source_id, target_id in sorted(candidates):
            if source_id not in used_sources and target_id not in used_targets:
                edges.append({'source_id': source_id, 'target_id': target_id})
                used_sources.add(source_id)
                used_targets.add(target_id)
    return edges


def dataset_name(path):
    name = path.name[:-5] if path.name.endswith('.zarr') else path.stem
    return name[:-7] if name.endswith('_labels') else name


def find_label_stores():
    stores = set()
    for root in SEARCH_ROOTS:
        if root.exists():
            stores.update(path for path in root.rglob('*.zarr') if path.is_dir())
            stores.update(path for path in root.rglob('*_labels.npz') if path.is_file())
            stores.update(path for path in root.rglob('*_labels.npy') if path.is_file())
    if not stores:
        raise FileNotFoundError('No .zarr, *_labels.npz, or *_labels.npy test stores were found.')
    return sorted(stores, key=str)


def generate_submission(stores, output_path):
    all_nodes = []
    all_edges = []
    next_node_id = 1
    for store in stores:
        dataset = dataset_name(store)
        nodes = extract_nodes(load_labels(store))
        for node in nodes:
            node['node_id'] = next_node_id
            next_node_id += 1
        all_nodes.extend((dataset, node) for node in nodes)
        all_edges.extend((dataset, edge) for edge in link_nodes(nodes))
        print(f'{dataset}: {len(nodes)} nodes')
    fields = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    with output_path.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        row_id = 0
        for dataset, node in all_nodes:
            writer.writerow({'id': row_id, 'dataset': dataset, 'row_type': 'node', **node, 'source_id': -1, 'target_id': -1})
            row_id += 1
        for dataset, edge in all_edges:
            writer.writerow({'id': row_id, 'dataset': dataset, 'row_type': 'edge', 'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1, **edge})
            row_id += 1
    return len(all_nodes), len(all_edges)


label_stores = find_label_stores()
print('Discovered label stores:')
for path in label_stores:
    print(path)
print(f'Output: {OUTPUT_PATH}')

Discovered label stores:
/Users/manmeet/Desktop/Bio Hub Project/kagglehub/kalman_labels.npz
/Users/manmeet/Desktop/Bio Hub Project/kagglehub/test2_labels.npz
/Users/manmeet/Desktop/Bio Hub Project/kagglehub/test_labels.npz
/Users/manmeet/Desktop/Bio Hub Project/kagglehub/train_labels.npz
Output: /Users/manmeet/Desktop/Bio Hub Project/kagglehub/submission.csv


In [ ]:
node_count, edge_count = generate_submission(label_stores, OUTPUT_PATH)
print(f'Created {OUTPUT_PATH} with {node_count} nodes and {edge_count} edges.')

Prepared kalman: 3 nodes
Prepared test2: 2 nodes
Prepared test: 2 nodes
Prepared train: 2 nodes
Wrote /Users/manmeet/Desktop/Bio Hub Project/kagglehub/submission.csv with 9 nodes and 5 edges
Created /Users/manmeet/Desktop/Bio Hub Project/kagglehub/submission.csv


In [ ]:
import csv

required = {'id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'}
with OUTPUT_PATH.open(newline='') as handle:
    rows = list(csv.DictReader(handle))

if not rows:
    raise ValueError('Submission is empty.')
if set(rows[0]) != required:
    raise ValueError(f'Unexpected columns: {set(rows[0])}')
if [row['id'] for row in rows] != [str(index) for index in range(len(rows))]:
    raise ValueError('The id column must be consecutive.')
node_ids = {row['node_id'] for row in rows if row['row_type'] == 'node'}
for row in rows:
    if not row['dataset']:
        raise ValueError('Every row must include a dataset name.')
    if row['row_type'] == 'node':
        for field in ('t', 'z', 'y', 'x'):
            if not float(row[field]).is_integer():
                raise ValueError(f'Node coordinate {field} is not an integer.')
    elif row['row_type'] == 'edge':
        if row['source_id'] not in node_ids or row['target_id'] not in node_ids:
            raise ValueError('Edge references an unknown node.')
    else:
        raise ValueError(f'Unsupported row type: {row["row_type"]}')

print(f'Validated {len(rows)} rows across {len({row["dataset"] for row in rows})} datasets.')
print(f'Upload this file: {OUTPUT_PATH}')